In [3]:
import psycopg2
from psycopg2 import OperationalError
from abc import ABC, abstractmethod
import tkinter as tk
from tkinter import ttk, messagebox
import os

# Interfaz base de Contenido
class Contenido(ABC):
    @abstractmethod
    def mostrar_contenido(self):
        pass

# Clase Video
class Video(Contenido):
    def __init__(self, titulo, descripcion, ruta):
        # Reemplazar las barras invertidas (\) por barras normales (/)
        self.titulo = titulo
        self.descripcion = descripcion
        self.ruta = ruta.replace("\\", "/")

    def mostrar_contenido(self):
        print(f"Video: {self.titulo} - {self.descripcion} ({self.ruta})")

# Clase Categoria
class Categoria(Contenido):
    def __init__(self, nombre, categoria_padre_nombre=None):
        self.nombre = nombre
        self.categoria_padre_nombre = categoria_padre_nombre
        self.contenidos = []

    def agregar(self, contenido):
        self.contenidos.append(contenido)

    def eliminar(self, contenido):
        self.contenidos.remove(contenido)

    def mostrar_contenido(self):
        print(f"Categoría: {self.nombre}")
        for contenido in self.contenidos:
            contenido.mostrar_contenido()

# Clase VideoDB para manejar PostgreSQL con validación de conexión
class VideoDB:
    def __init__(self, db_name, user, password, host="localhost", port="5432"):
        try:
            # Intentar la conexión a PostgreSQL
            self.conn = psycopg2.connect(
                dbname=db_name,
                user=user,
                password=password,
                host=host,
                port=port
            )
            self.cursor = self.conn.cursor()
            self.inicializar_tablas()
            print("Conexión exitosa a la base de datos PostgreSQL.")
        except OperationalError as e:
            print(f"Error al conectar a la base de datos: {e}")
            self.conn = None
            self.cursor = None

    def inicializar_tablas(self):
        if not self.conn:
            print("No se puede inicializar tablas: sin conexión a la base de datos.")
            return

        # Tabla Categoria
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS Categoria (
            nombre TEXT PRIMARY KEY,
            categoria_padre_nombre TEXT,
            FOREIGN KEY (categoria_padre_nombre) REFERENCES Categoria(nombre) ON DELETE CASCADE
        )""")
        
        # Tabla Video con campo extra para la ruta
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS Video (
            titulo TEXT PRIMARY KEY,
            descripcion TEXT,
            ruta TEXT
        )""")
        
        # Tabla intermedia VideoCategoria para relación muchos a muchos
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS VideoCategoria (
            video_titulo TEXT,
            categoria_nombre TEXT,
            PRIMARY KEY (video_titulo, categoria_nombre),
            FOREIGN KEY (video_titulo) REFERENCES Video(titulo) ON DELETE CASCADE,
            FOREIGN KEY (categoria_nombre) REFERENCES Categoria(nombre) ON DELETE CASCADE
        )""")
        
        self.conn.commit()

    def cargar_categorias(self):
        if not self.conn:
            print("Error: No hay conexión a la base de datos.")
            return None
        # Crear la categoría raíz
        raiz = Categoria("Raíz")
        # Cargar recursivamente todas las subcategorías
        self.cargar_subcategorias_recursivamente(raiz)
        return raiz

    def cargar_subcategorias_recursivamente(self, categoria):
        # Obtiene todas las subcategorías de la categoría actual
        subcategorias = self.cargar_subcategorias(categoria.nombre)
        for subcategoria in subcategorias:
            categoria.agregar(subcategoria)
            # Llama recursivamente para obtener las subcategorías de cada subcategoría
            self.cargar_subcategorias_recursivamente(subcategoria)

    def cargar_subcategorias(self, categoria_padre_nombre):
        if categoria_padre_nombre == "Raíz":
            # Cargar las categorías de nivel superior (las que no tienen categoria_padre_nombre)
            self.cursor.execute("SELECT nombre FROM Categoria WHERE categoria_padre_nombre IS NULL")
        else:
            # Cargar las subcategorías de una categoría específica
            self.cursor.execute("SELECT nombre FROM Categoria WHERE categoria_padre_nombre = %s", (categoria_padre_nombre,))
        
        return [Categoria(row[0], categoria_padre_nombre) for row in self.cursor.fetchall()]

    def cargar_videos_por_categoria(self, categoria_nombre):
        self.cursor.execute("""
        SELECT Video.titulo, Video.descripcion, Video.ruta
        FROM Video
        JOIN VideoCategoria ON Video.titulo = VideoCategoria.video_titulo
        WHERE VideoCategoria.categoria_nombre = %s
        """, (categoria_nombre,))
        # Normalizar las rutas de los videos al cargarlas desde la base de datos
        return [Video(row[0], row[1], row[2].replace("\\", "/")) for row in self.cursor.fetchall()]

    def agregar_categoria(self, nombre, categoria_padre_nombre=None):
        if not self.conn:
            print("Error: No hay conexión a la base de datos.")
            return
        try:
            self.cursor.execute("INSERT INTO Categoria (nombre, categoria_padre_nombre) VALUES (%s, %s)", (nombre, categoria_padre_nombre))
            self.conn.commit()
        except psycopg2.IntegrityError:
            print(f"La categoría '{nombre}' ya existe.")
            self.conn.rollback()

    def agregar_video(self, titulo, descripcion, ruta):
        if not self.conn:
            print("Error: No hay conexión a la base de datos.")
            return
        try:
            # Normalizar la ruta antes de almacenar
            ruta_normalizada = ruta.replace("\\", "/")
            self.cursor.execute("INSERT INTO Video (titulo, descripcion, ruta) VALUES (%s, %s, %s)", (titulo, descripcion, ruta_normalizada))
            self.conn.commit()
        except psycopg2.IntegrityError:
            print(f"El video '{titulo}' ya existe.")
            self.conn.rollback()

    def eliminar_video(self, titulo):
        """Elimina un video de la base de datos por su título."""
        if not self.conn:
            print("Error: No hay conexión a la base de datos.")
            return
        try:
            self.cursor.execute("DELETE FROM Video WHERE titulo = %s", (titulo,))
            self.conn.commit()
        except psycopg2.Error as e:
            print(f"Error al eliminar el video '{titulo}': {e}")
            self.conn.rollback()

    def agregar_video_categoria(self, video_titulo, categoria_nombre):
        if not self.conn:
            print("Error: No hay conexión a la base de datos.")
            return
        try:
            self.cursor.execute("INSERT INTO VideoCategoria (video_titulo, categoria_nombre) VALUES (%s, %s)", (video_titulo, categoria_nombre))
            self.conn.commit()
        except psycopg2.IntegrityError:
            print(f"El video '{video_titulo}' ya está asignado a la categoría '{categoria_nombre}'.")
            self.conn.rollback()

    def __del__(self):
        if self.conn:
            self.conn.close()

# Clase VideoApp para la interfaz gráfica
class VideoApp(tk.Tk):
    def __init__(self, db_name, user, password, host="localhost", port="5432"):
        super().__init__()
        self.title("VideoApp_Composite")
        self.geometry("2560x1440")
        
        # Conectar a la base de datos PostgreSQL
        self.video_db = VideoDB(db_name, user, password, host, port)
        
        if not self.video_db.conn:
            messagebox.showerror("Error de conexión", "No se pudo conectar a la base de datos.")
            self.destroy()  # Cierra la aplicación si no hay conexión
            return

        # Cargar la jerarquía completa desde la base de datos
        self.raiz = self.video_db.cargar_categorias()
        if not self.raiz:
            messagebox.showerror("Error de carga", "No se pudieron cargar las categorías.")

        # Crear árbol de categorías
        self.category_tree = ttk.Treeview(self)
        self.category_tree.pack(fill=tk.BOTH, expand=True)
        self.carga_categoria_arbol()

        # Conectar el evento de selección en el árbol
        self.category_tree.bind("<<TreeviewSelect>>", self.seleccion_en_arbol)

        # Área de entrada para agregar categorías y videos
        self.crea_area_entrada()

        # Lista para mostrar los videos de una categoría seleccionada
        self.video_listbox = tk.Listbox(self)
        self.video_listbox.pack(fill=tk.BOTH, expand=True)
        self.video_listbox.bind("<<ListboxSelect>>", self.seleccion_en_video)

        # Botón para eliminar video
        self.delete_button = tk.Button(self, text="Eliminar Video", command=self.elimina_video)
        self.delete_button.pack()

        # Variable para almacenar el video seleccionado
        self.selected_video = None

    def carga_categoria_arbol(self):
        root_item = self.category_tree.insert("", "end", text=self.raiz.nombre, open=True)
        self.crea_arbol(root_item, self.raiz)

    def crea_arbol(self, parent_item, categoria):
        for contenido in categoria.contenidos:
            if isinstance(contenido, Categoria):
                item = self.category_tree.insert(parent_item, "end", text=contenido.nombre, open=True)
                self.crea_arbol(item, contenido)
            elif isinstance(contenido, Video):
                self.category_tree.insert(parent_item, "end", text=contenido.titulo)

    def crea_area_entrada(self):
        # Entrada para agregar categorías
        tk.Label(self, text="Nombre de Categoría").pack()
        self.category_name_entry = tk.Entry(self,width=100)
        self.category_name_entry.pack()
        tk.Button(self, text="Agregar Categoría", command=self.agrega_categoria).pack()

        # Entrada para agregar videos
        tk.Label(self, text="Título de Video").pack()
        self.video_title_entry = tk.Entry(self,width=100)
        self.video_title_entry.pack()
        
        tk.Label(self, text="Descripción").pack()
        self.video_description_entry = tk.Entry(self,width=100)
        self.video_description_entry.pack()
        
        tk.Label(self, text="Ruta del Video").pack()
        self.video_path_entry = tk.Entry(self,width=100)
        self.video_path_entry.pack()
        
        tk.Button(self, text="Agregar Video", command=self.agrega_video).pack()

    def agrega_categoria(self):
        category_name = self.category_name_entry.get()
        selected_item = self.category_tree.selection()
        
        if selected_item:
            parent_category_name = self.category_tree.item(selected_item, "text")
            self.video_db.agregar_categoria(category_name, parent_category_name)
            parent_category = self.buscar_categoria(parent_category_name, self.raiz)
            if parent_category:
                nueva_categoria = Categoria(category_name, parent_category_name)
                parent_category.agregar(nueva_categoria)
                self.category_tree.insert(selected_item, "end", text=category_name)

    def agrega_video(self):
        video_title = self.video_title_entry.get()
        video_description = self.video_description_entry.get()
        video_path = self.video_path_entry.get()
        full_path = video_path.replace("\\", "/")  # Normalizar la ruta a usar `/`
        selected_item = self.category_tree.selection()
        
        if selected_item:
            # Verificar la categoría seleccionada para asignar el video correctamente
            category_name = self.category_tree.item(selected_item, "text")
            self.video_db.agregar_video(video_title, video_description, full_path)
            self.video_db.agregar_video_categoria(video_title, category_name)

            # Actualizar la lista de videos en la interfaz gráfica
            self.video_listbox.insert(tk.END, video_title)

    def seleccion_en_arbol(self, event):
        selected_item = self.category_tree.selection()[0]
        selected_text = self.category_tree.item(selected_item, "text")
        
        # Limpiar la lista de videos
        self.video_listbox.delete(0, tk.END)
        
        # Cargar y mostrar videos de la categoría seleccionada
        videos = self.video_db.cargar_videos_por_categoria(selected_text)
        for video in videos:
            self.video_listbox.insert(tk.END, video.titulo)

    def seleccion_en_video(self, event):
        selection = self.video_listbox.curselection()
        if selection:
            index = selection[0]
            video_title = self.video_listbox.get(index)
            # Buscar el video seleccionado en la base de datos
            videos = self.video_db.cargar_videos_por_categoria(self.category_tree.item(self.category_tree.selection()[0], "text"))
            for video in videos:
                if video.titulo == video_title:
                    self.selected_video = video
                    self.muestra_detalles_video(video)
                    break

    def muestra_detalles_video(self, video):
        # Crear una nueva ventana para mostrar los detalles del video
        details_window = tk.Toplevel(self)
        details_window.title(video.titulo)
        tk.Label(details_window, text=video.descripcion, wraplength=300).pack()
        
        play_button = tk.Button(details_window, text="Reproducir Video", command=lambda: self.play_video(video))
        play_button.pack()

    def play_video(self, video):
        if video.ruta:
            # Intenta abrir el video en el reproductor predeterminado
            os.startfile(video.ruta)

    def elimina_video(self):
        # Eliminar el video seleccionado
        selection = self.video_listbox.curselection()
        if selection:
            index = selection[0]
            video_title = self.video_listbox.get(index)
            confirm = messagebox.askyesno("Confirmar", f"¿Deseas eliminar el video '{video_title}'?")
            if confirm:
                self.video_db.eliminar_video(video_title)
                self.video_listbox.delete(index)
                self.selected_video = None
                messagebox.showinfo("Eliminado", f"El video '{video_title}' ha sido eliminado.")

    def buscar_categoria(self, nombre, categoria):
        if categoria.nombre == nombre:
            return categoria
        for contenido in categoria.contenidos:
            if isinstance(contenido, Categoria):
                resultado = self.buscar_categoria(nombre, contenido)
                if resultado:
                    return resultado
        return None
if __name__ == "__main__":
    # Conexión de la app a PostgreSQL, reemplaza con tus credenciales
    app = VideoApp(db_name="Videos", user="postgres", password="ul1se3ol4")
    app.mainloop()


Conexión exitosa a la base de datos PostgreSQL.


Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Program Files\Python311\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Rulig\AppData\Local\Temp\ipykernel_19896\155533864.py", line 322, in <lambda>
    play_button = tk.Button(details_window, text="Reproducir Video", command=lambda: self.play_video(video))
                                                                                     ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Rulig\AppData\Local\Temp\ipykernel_19896\155533864.py", line 328, in play_video
    os.startfile(video.ruta)
FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado: 'C:/Users/Rulig/OneDrive/Escritorio/apps/progra/UNAM/Modelado y programacion/expo/implementacion/Oppenheimer.2023.1080P-Dual-Lat.mp4'
Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Program Files\Python311\Lib\tkinter\__init__.py", line 1948,

El video 'Oppenheimer' ya existe.
El video 'Oppenheimer' ya está asignado a la categoría 'Drama'.
El video 'Oppenheimer' ya existe.
